# Estimate measurements from .csv files 

In [1]:
import os
import pandas as pd
import re
from pathlib import Path
import time

def count_csv_rows_fast(csv_path):
    """
    Fast row counting method using line counting instead of pandas loading.
    This is much faster for large files since we don't need to parse the data.
    """
    try:
        with open(csv_path, 'r', encoding='utf-8', errors='ignore') as f:
            # Count lines and subtract 1 for header (assuming CSV has headers)
            row_count = sum(1 for _ in f) - 1
            return max(0, row_count)  # Ensure non-negative count
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")
        return 0

def parse_filename(filename):
    """
    Parse the filename to extract doc_id, page_number, and table_number.
    Expected format: 'docid_page_pagenumber_tabletablenumber.csv'
    Example: '431_page_11_table1.csv'
    """
    # Remove .csv extension
    name_without_ext = filename.replace('.csv', '')
    
    # Use regex to parse the pattern
    pattern = r'(\d+)_page_(\d+)_table(\d+)'
    match = re.match(pattern, name_without_ext)
    
    if match:
        doc_id = int(match.group(1))
        page_number = int(match.group(2))
        table_number = int(match.group(3))
        return doc_id, page_number, table_number
    else:
        print(f"Warning: Could not parse filename: {filename}")
        return None, None, None

def process_single_csv(csv_path):
    """
    Process a single CSV file and return its information.
    """
    filename = os.path.basename(csv_path)
    doc_id, page_number, table_number = parse_filename(filename)
    
    if doc_id is not None:
        row_count = count_csv_rows_fast(csv_path)
        return {
            'doc_id': doc_id,
            'page_number': page_number,
            'table_number': table_number,
            'number_rows': row_count
        }
    return None

def main():
    # Configuration
    main_folder = r"C:\Users\aeliz\Dropbox\Chap2_USGS\ReductCSVs"
    output_file = r"C:\Users\aeliz\Dropbox\Chap2_USGS\csv_row_counts.csv"
    
    # Get all document folders
    doc_folders = [f for f in Path(main_folder).iterdir() if f.is_dir()]
    print(f"Found {len(doc_folders)} document folders")
    
    # Start timing
    start_time = time.time()
    
    all_results = []
    processed_folders = 0
    processed_files = 0
    
    # Process folders sequentially (more stable)
    for doc_folder in doc_folders:
        try:
            # Get all CSV files in this folder
            csv_files = list(Path(doc_folder).glob('*.csv'))
            
            # Process each CSV file
            for csv_file in csv_files:
                result = process_single_csv(csv_file)
                if result:
                    all_results.append(result)
                    processed_files += 1
            
            processed_folders += 1
            
            # Progress update every 50 folders
            if processed_folders % 50 == 0:
                elapsed = time.time() - start_time
                files_per_minute = processed_files / (elapsed / 60) if elapsed > 0 else 0
                estimated_total_time = (len(doc_folders) * elapsed / processed_folders) / 60
                
                print(f"Progress: {processed_folders}/{len(doc_folders)} folders "
                      f"({processed_folders/len(doc_folders)*100:.1f}%) | "
                      f"Files processed: {processed_files:,} | "
                      f"Speed: {files_per_minute:.0f} files/min | "
                      f"Elapsed: {elapsed/60:.1f}min | "
                      f"Est. total time: {estimated_total_time:.1f}min")
                
                # Save intermediate results every 1000 folders to avoid losing progress
                if processed_folders % 1000 == 0:
                    temp_df = pd.DataFrame(all_results)
                    temp_df = temp_df.sort_values(['doc_id', 'page_number', 'table_number'])
                    temp_output = output_file.replace('.csv', f'_temp_{processed_folders}.csv')
                    temp_df.to_csv(temp_output, index=False)
                    print(f"Saved intermediate results to: {temp_output}")
                    
        except Exception as e:
            print(f"Error processing folder {doc_folder}: {e}")
            continue
    
    # Convert to DataFrame and save final results
    if all_results:
        print(f"\nProcessing complete! Creating final output...")
        df = pd.DataFrame(all_results)
        
        # Sort by doc_id, page_number, table_number for better organization
        df = df.sort_values(['doc_id', 'page_number', 'table_number'])
        
        # Save to CSV
        df.to_csv(output_file, index=False)
        
        # Print summary statistics
        end_time = time.time()
        elapsed_minutes = (end_time - start_time) / 60
        
        print(f"\n=== SUMMARY ===")
        print(f"Total CSV files processed: {len(all_results):,}")
        print(f"Total document folders processed: {processed_folders:,}")
        print(f"Total processing time: {elapsed_minutes:.1f} minutes")
        print(f"Average files per minute: {len(all_results)/elapsed_minutes:.0f}")
        print(f"Output saved to: {output_file}")
        
        # Show some sample statistics
        print(f"\nRow count statistics:")
        print(f"Total rows across all CSVs: {df['number_rows'].sum():,}")
        print(f"Average rows per CSV: {df['number_rows'].mean():.1f}")
        print(f"Median rows per CSV: {df['number_rows'].median():.1f}")
        print(f"Max rows in a single CSV: {df['number_rows'].max():,}")
        print(f"Files with 0 rows: {(df['number_rows'] == 0).sum()}")
        
        # Show first few rows
        print(f"\nFirst 10 results:")
        print(df.head(10).to_string(index=False))
        
        # Show distribution of document IDs
        doc_counts = df['doc_id'].value_counts().head(10)
        print(f"\nTop 10 documents by number of CSV files:")
        for doc_id, count in doc_counts.items():
            print(f"Document {doc_id}: {count} CSV files")
            
    else:
        print("No CSV files were successfully processed!")
        print("Please check that the folder path is correct and contains CSV files.")

if __name__ == "__main__":
    main()

Found 622 document folders
Progress: 50/622 folders (8.0%) | Files processed: 3,861 | Speed: 4754 files/min | Elapsed: 0.8min | Est. total time: 10.1min
Progress: 100/622 folders (16.1%) | Files processed: 20,846 | Speed: 4559 files/min | Elapsed: 4.6min | Est. total time: 28.4min
Progress: 150/622 folders (24.1%) | Files processed: 29,899 | Speed: 4533 files/min | Elapsed: 6.6min | Est. total time: 27.4min
Progress: 200/622 folders (32.2%) | Files processed: 33,098 | Speed: 4537 files/min | Elapsed: 7.3min | Est. total time: 22.7min
Progress: 250/622 folders (40.2%) | Files processed: 34,756 | Speed: 4541 files/min | Elapsed: 7.7min | Est. total time: 19.0min
Progress: 300/622 folders (48.2%) | Files processed: 42,854 | Speed: 4504 files/min | Elapsed: 9.5min | Est. total time: 19.7min
Progress: 350/622 folders (56.3%) | Files processed: 51,020 | Speed: 4461 files/min | Elapsed: 11.4min | Est. total time: 20.3min
Progress: 400/622 folders (64.3%) | Files processed: 55,743 | Speed: 450

## SB Data additions 

In [ ]:
import os
import pandas as pd
import re
from pathlib import Path
import time

def count_csv_rows_fast(csv_path):
    """
    Fast row counting method using line counting instead of pandas loading.
    This is much faster for large files since we don't need to parse the data.
    """
    try:
        with open(csv_path, 'r', encoding='utf-8', errors='ignore') as f:
            # Count lines and subtract 1 for header (assuming CSV has headers)
            row_count = sum(1 for _ in f) - 1
            return max(0, row_count)  # Ensure non-negative count
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")
        return 0

def parse_filename(filename):
    """
    Parse the filename to extract doc_id, page_number, and table_number.
    Expected format: 'docid_page_pagenumber_tabletablenumber.csv'
    Example: '431_page_11_table1.csv'
    """
    # Remove .csv extension
    name_without_ext = filename.replace('.csv', '')
    
    # Use regex to parse the pattern
    pattern = r'(\d+)_page_(\d+)_table(\d+)'
    match = re.match(pattern, name_without_ext)
    
    if match:
        doc_id = int(match.group(1))
        page_number = int(match.group(2))
        table_number = int(match.group(3))
        return doc_id, page_number, table_number
    else:
        print(f"Warning: Could not parse filename: {filename}")
        return None, None, None

def process_single_csv(csv_path):
    """
    Process a single CSV file and return its information.
    """
    filename = os.path.basename(csv_path)
    doc_id, page_number, table_number = parse_filename(filename)
    
    if doc_id is not None:
        row_count = count_csv_rows_fast(csv_path)
        return {
            'doc_id': doc_id,
            'page_number': page_number,
            'table_number': table_number,
            'number_rows': row_count
        }
    return None

def main():
    # Configuration
    main_folder = r"C:\Users\aeliz\Dropbox\Chap2_USGS\ReductCSVs"
    output_file = r"C:\Users\aeliz\Dropbox\Chap2_USGS\csv_row_counts.csv"
    
    # Get all document folders
    doc_folders = [f for f in Path(main_folder).iterdir() if f.is_dir()]
    print(f"Found {len(doc_folders)} document folders")
    
    # Start timing
    start_time = time.time()
    
    all_results = []
    processed_folders = 0
    processed_files = 0
    
    # Process folders sequentially (more stable)
    for doc_folder in doc_folders:
        try:
            # Get all CSV files in this folder
            csv_files = list(Path(doc_folder).glob('*.csv'))
            
            # Process each CSV file
            for csv_file in csv_files:
                result = process_single_csv(csv_file)
                if result:
                    all_results.append(result)
                    processed_files += 1
            
            processed_folders += 1
            
            # Progress update every 50 folders
            if processed_folders % 50 == 0:
                elapsed = time.time() - start_time
                files_per_minute = processed_files / (elapsed / 60) if elapsed > 0 else 0
                estimated_total_time = (len(doc_folders) * elapsed / processed_folders) / 60
                
                print(f"Progress: {processed_folders}/{len(doc_folders)} folders "
                      f"({processed_folders/len(doc_folders)*100:.1f}%) | "
                      f"Files processed: {processed_files:,} | "
                      f"Speed: {files_per_minute:.0f} files/min | "
                      f"Elapsed: {elapsed/60:.1f}min | "
                      f"Est. total time: {estimated_total_time:.1f}min")
                
                # Save intermediate results every 1000 folders to avoid losing progress
                if processed_folders % 1000 == 0:
                    temp_df = pd.DataFrame(all_results)
                    temp_df = temp_df.sort_values(['doc_id', 'page_number', 'table_number'])
                    temp_output = output_file.replace('.csv', f'_temp_{processed_folders}.csv')
                    temp_df.to_csv(temp_output, index=False)
                    print(f"Saved intermediate results to: {temp_output}")
                    
        except Exception as e:
            print(f"Error processing folder {doc_folder}: {e}")
            continue
    
    # Convert to DataFrame and save final results
    if all_results:
        print(f"\nProcessing complete! Creating final output...")
        df = pd.DataFrame(all_results)
        
        # Sort by doc_id, page_number, table_number for better organization
        df = df.sort_values(['doc_id', 'page_number', 'table_number'])
        
        # Save to CSV
        df.to_csv(output_file, index=False)
        
        # Print summary statistics
        end_time = time.time()
        elapsed_minutes = (end_time - start_time) / 60
        
        print(f"\n=== SUMMARY ===")
        print(f"Total CSV files processed: {len(all_results):,}")
        print(f"Total document folders processed: {processed_folders:,}")
        print(f"Total processing time: {elapsed_minutes:.1f} minutes")
        print(f"Average files per minute: {len(all_results)/elapsed_minutes:.0f}")
        print(f"Output saved to: {output_file}")
        
        # Show some sample statistics
        print(f"\nRow count statistics:")
        print(f"Total rows across all CSVs: {df['number_rows'].sum():,}")
        print(f"Average rows per CSV: {df['number_rows'].mean():.1f}")
        print(f"Median rows per CSV: {df['number_rows'].median():.1f}")
        print(f"Max rows in a single CSV: {df['number_rows'].max():,}")
        print(f"Files with 0 rows: {(df['number_rows'] == 0).sum()}")
        
        # Show first few rows
        print(f"\nFirst 10 results:")
        print(df.head(10).to_string(index=False))
        
        # Show distribution of document IDs
        doc_counts = df['doc_id'].value_counts().head(10)
        print(f"\nTop 10 documents by number of CSV files:")
        for doc_id, count in doc_counts.items():
            print(f"Document {doc_id}: {count} CSV files")
            
    else:
        print("No CSV files were successfully processed!")
        print("Please check that the folder path is correct and contains CSV files.")

if __name__ == "__main__":
    main()